# Mercado Automotriz de La Habana — Informe final

**Proyecto de Ciencia de Datos · 1er año · Universidad de La Habana**

**Fase 6 · Informe final del proyecto**

**Fecha:** 24 de septiembre de 2026

Este documento integra las fases completadas del proyecto — esquema de datos,
scraping, limpieza, análisis exploratorio (EDA) y modelo de precios — sobre la
**muestra ampliada de 600 anuncios** de Revolico (La Habana). Todas las cifras
citadas provienen de ejecuciones reales: los datasets de `data/processed/`, las
tablas y figuras de `reports/`, y los notebooks `01_eda.ipynb` y
`02_modelo_precios.ipynb`.


## Resumen ejecutivo

Con los resultados reales de la muestra ampliada (scraping 150 → 600 anuncios):

1. Se scrapearon **600 anuncios** de carros en La Habana a través de la API
   GraphQL pública de Revolico (≈17% del total publicado: 3 517), con respaldo
   JSON y sin datos de contacto.
2. **555 anuncios publican precio (92%)**; la mediana es de **19 500 USD** y la
   distribución es muy dispersa (de precios simbólicos de 1 USD hasta 250 000 USD).
3. El mercado es **mayoritariamente moderno**: 94% de los anuncios con año son
   posteriores a 2010.
4. El **año es el factor más explicativo** de los disponibles: **+716 USD por año
   adicional** (r = 0,30 entre año y precio).
5. La **regresión lineal** (M0–M3, split 75/25 + validación cruzada 5-fold) mejora
   notablemente con más datos: R² de test pasa de negativo (−0.94, −0.88 en la
   muestra chica) a **+0.04 y +0.03** en M1 y M2; sin embargo, la CV sigue
   negativa: **el modelo no supera consistentemente a predecir la media**.
6. La causa principal **ya no es el tamaño de muestra** sino la **ausencia de
   variables fuertes** en Revolico (km, estado, combustible y transmisión están
   vacías en el 100% de los anuncios).
7. **Conclusión Q1 (cualitativa):** en estos datos, **año y marca** son las únicas
   variables con señal detectable; la zona y las vistas no aportan poder
   predictivo. El trabajo futuro apunta a la recolección de campo (Fase 3).


## 1. Introducción

No existe un dataset público del mercado automovilístico cubano, por lo que el
proyecto **construye sus propios datos** a partir de anuncios publicados en
Revolico (el mercado «declarado»). Las preguntas de investigación son:

- **Q1.** ¿Qué factores explican el precio de un carro en La Habana?
  *(año, marca, zona, visibilidad…)*
- **Q2.** ¿Los precios publicados difieren de la realidad observada en la calle?
  *(requiere recolección de campo — Fase 3, en pausa por decisión del equipo)*
- **Q3.** ¿Qué segmentos de mercado existen? *(clásicos, modernos, económicos…)*

La Fase 6 sintetiza el camino recorrido, los resultados verificados y las
limitaciones, y responde Q1 y Q3 con evidencia de las Fases 4 y 5.


## 2. Metodología

### 2.1 Fuente de datos y scraping (Fase 2)

Revolico es un sitio **Next.js/React detrás de Cloudflare**: su HTML no contiene
los anuncios (se cargan por JavaScript), por lo que `requests + BeautifulSoup` a
secas no sirve. Mediante ingeniería inversa de los *bundles* JS se descubrió la
**API GraphQL pública** `https://graphql-api.revolico.app/` (proceso documentado en
`src/exploracion/`), con la query `AdsSearch`.

Filtros usados: subcategoría `vehiculos-carros` y provincia `la-habana`, con
paginación `page` + `pageLength` (máximo 50 por página).

- **Scraping original:** 3 páginas × 50 = 150 anuncios únicos.
- **Ampliación (revisión de la Fase 5):** 12 páginas × 50 = **600 anuncios
  únicos**, deduplicados por `anuncio_id` (0 duplicados finales).
- Total publicado según la API: **3 517** → la muestra cubre ≈17%.
- **Buenas prácticas:** pausa de 4 s entre páginas, respaldo JSON crudo por
  página, **no se guardan teléfonos ni datos de contacto** (decisión ética).
- La muestra original (150) quedó respaldada en
  `data/raw/respaldo_150_2026-09-24/`.

Campos guardados: fuente, fecha, ubicación (municipio), marca, modelo, año
(parcial, parseado del título), precio original, moneda, precio en USD,
descripción, URL, `anuncio_id`, vistas, número de imágenes, destacado y fecha de
actualización.


### 2.2 Limpieza (Fase 4)

Decisiones acordadas y aplicadas en `notebooks/01_eda.ipynb`:

1. **Años faltantes → `anio = 0`.** No se eliminan del dataset; solo se excluyen
   de los análisis que necesitan año. En la muestra ampliada, 279 de 600 (46%)
   quedaron con `anio = 0`.
2. **Moneda:** tasa de referencia **710 CUP = 1 USD** (constante `TASA_CUP_USD`)
   → los 6 anuncios en CUP se convirtieron con `precio_usd = precio_cup / 710`.
3. **Marca no identificada** (210 anuncios, 35%) se agrupa como `no identificada`
   y se excluye solo de los análisis por marca; los modelos ruidosos de los
   títulos se corrigieron con un diccionario de marcas.
4. **Precios simbólicos** (15 anuncios a 1 USD) se conservan; los **3 valores
   atípicos altos** detectados por IQR (límite 79 500 USD: un dongfeng a 82 000 y
   dos BMW X6 a 120 000 y 250 000 USD) se documentan sin eliminar.

Resultado: `data/processed/mercado_havana.csv` con 600 filas y 17 columnas útiles.


### 2.3 Modelo de precios (Fase 5)

En `notebooks/02_modelo_precios.ipynb` se aplicó el pipeline correcto de
regresión, auditado para evitar *data leakage*:

1. **Excluir por modelo:** cada especificación usa solo sus filas completas y
   reporta su `n` (no se eliminan filas del dataset): **M0 n=555** (con precio),
   **M1 n=300** (+ año), **M2 y M3 n=247** (+ año y marca).
2. **Dos variantes del target:** `precio_usd` (coeficientes en USD, interpretable)
   y `log1p(precio_usd)` (chequeo de robustez frente a los valores extremos).
3. **Split 75/25 con `random_state = 42`** → el One-Hot de marcas se ajusta
   **solo en train** → se entrena en train → se evalúa en test.
4. **Validación cruzada 5-fold** como chequeo de robustez frente a la volatilidad
   del split único.

Modelos: **M0** (predecir la media, baseline), **M1** (`precio ~ año`),
**M2** (`precio ~ año + marca`), **M3** (`precio ~ año + marca + municipio +
vistas + imágenes`).


### 2.4 Retrato de la muestra (Tabla 1)

Los números que sustentan el informe se recalculan aquí desde el dataset limpio
real (`data/processed/mercado_havana.csv`), de modo que el texto cite salidas de
ejecución y no valores escritos a mano.


In [1]:
# Tabla 1 · Retrato de la muestra (leído del CSV real)
import pandas as pd
import numpy as np
from pathlib import Path

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

df = pd.read_csv(RAIZ / "data/processed/mercado_havana.csv")
p = df.dropna(subset=["precio_usd"])
con_anio = df[df["anio"] > 0]
pa = p[p["anio"] > 0]  # con precio y año

print(f"Anuncios únicos:            {len(df)}")
print(f"Con precio:                 {len(p)} ({len(p) / len(df):.0%})")
print(f"  - en CUP (710 CUP/USD):   {(df['moneda'] == 'CUP').sum()}")
print(f"Sin precio ('a convenir'):  {len(df) - len(p)}")
print(f"Con año:                    {len(con_anio)} ({len(con_anio) / len(df):.0%})")
print(f"  - posteriores a 2010:     {(con_anio['anio'] > 2010).sum()} de {len(con_anio)} ({(con_anio['anio'] > 2010).sum() / len(con_anio):.0%})")
print(f"Marca no identificada:      {(df['marca'] == 'no identificada').sum()} ({(df['marca'] == 'no identificada').sum() / len(df):.0%})")
print(f"Municipios representados:   {df['municipio'].nunique()}  (Playa: {(df['municipio'] == 'Playa').sum()} = {(df['municipio'] == 'Playa').mean():.0%})")
print()
print(f"Precio (USD):  mediana {p['precio_usd'].median():,.0f} · media {p['precio_usd'].mean():,.0f} · desv. típica {p['precio_usd'].std():,.0f}")
print(f"  rango: {p['precio_usd'].min():.4f} – {p['precio_usd'].max():,.0f}  |  simbólicos a 1 USD: {(p['precio_usd'] == 1).sum()}")
print()
coef, _ = np.polyfit(pa["anio"], pa["precio_usd"], 1)
r = np.corrcoef(pa["anio"], pa["precio_usd"])[0, 1]
print(f"Tendencia año→precio: +{coef:,.0f} USD/año  (n = {len(pa)}, r = {r:.2f})")
m2010 = pa[(pa["anio"] >= 2000) & (pa["anio"] < 2010)]["precio_usd"].median()
m2020 = pa[pa["anio"] >= 2020]["precio_usd"].median()
print(f"Mediana 2000–2009: {m2010:,.0f} USD  ·  mediana 2020+: {m2020:,.0f} USD")


Anuncios únicos:            600
Con precio:                 555 (92%)
  - en CUP (710 CUP/USD):   6
Sin precio ('a convenir'):  45
Con año:                    321 (54%)
  - posteriores a 2010:     301 de 321 (94%)
Marca no identificada:      210 (35%)
Municipios representados:   15  (Playa: 189 = 32%)

Precio (USD):  mediana 19,500 · media 20,646 · desv. típica 19,505
  rango: 0.0014 – 250,000  |  simbólicos a 1 USD: 15

Tendencia año→precio: +716 USD/año  (n = 300, r = 0.30)
Mediana 2000–2009: 11,000 USD  ·  mediana 2020+: 34,000 USD


## 3. Resultados del análisis exploratorio (Fase 4)

### 3.1 Precios y oferta

La muestra está dominada por la gama media: la **mediana es de 19 500 USD**, con
una cola larga hacia los 250 000 USD del BMW X6 y 15 precios simbólicos de 1 USD.
El 92% de los anuncios publica precio. En cuanto a la oferta, **Hyundai domina con
137 anuncios (23%)**, seguida de Kia (33), Toyota (32), Nissan (31) y Lada (28).

![Distribución de precios](../reports/fig_01_precios.png)

![Anuncios por marca](../reports/fig_02_anuncios_por_marca.png)


### 3.2 Precio por marca y año

Se distinguen **dos segmentos por precio**: el medio-alto (mediana Toyota
38 000 USD, Nissan 32 000, Hyundai 25 000, Kia 23 500) y el económico/de trabajo
(Lada 6 000, Daewoo 11 000). El **año es la relación más fuerte del dataset**:
tendencia de **+716 USD por año adicional** (300 anuncios con año y precio,
r = 0,30); la mediana pasa de unos 11 000 USD en los carros de 2000–2009 a
~34 000 USD en los de 2020+.

![Precio por marca](../reports/fig_03_precio_por_marca.png)

![Precio por año](../reports/fig_04_precio_por_anio.png)


### 3.3 Décadas, municipios y visibilidad

El mercado es **mayoritariamente moderno**: 94% de los anuncios con año son de
2010 o posterior (solo 6 clásicos ≤1990, el más antiguo de 1952). La **zona pesa
poco**: Playa concentra la mayor oferta (189 anuncios, 32%) pero con mediana de
24 000 USD, por debajo de Arroyo Naranjo (26 000 USD). Las **vistas son bajas**
(mediana 13) y su correlación con el precio es débil y negativa (r ≈ −0,20): la
visibilidad no explica el precio.

![Anuncios por década](../reports/fig_05_por_decada.png)

![Precio por municipio](../reports/fig_06_precio_por_municipio.png)

![Precio vs. vistas](../reports/fig_07_precio_vs_vistas.png)


### 3.4 Hallazgos del EDA (respuesta preliminar a Q1 y Q3)

1. **Q3 · Oferta de carros modernos.** 94% de los anuncios con año son 2010+;
   Hyundai, Kia, Toyota, Nissan y Lada concentran la oferta.
2. **Q1 · Precio disperso.** Mediana 19 500 USD, rango de 1 a 250 000 USD; 3
   valores atípicos altos por IQR (BMW X6 ×2 y dongfeng).
3. **Q1 · El año es el factor más fuerte.** +716 USD/año (r = 0,30); mediana
   2000–2009 ≈ 11 000 USD vs. 2020+ ≈ 34 000 USD.
4. **Q3 · Segmentos por precio.** Medio-alto (Toyota/Nissan/Hyundai/Kia) vs.
   económico (Lada/Daewoo); la zona no discrimina entre segmentos.
5. **Límites de la muestra.** 35% sin marca, 46% sin año; vistas (r ≈ −0,20) no
   explican el precio. Los resultados se formalizan con la regresión de la Fase 5.


## 4. Resultados del modelo de precios (Fase 5)

### 4.1 Métricas del split 75/25 (Tabla 2)

La tabla siguiente se lee directamente de `reports/tabla_metricas_modelos.csv`
(generado por `02_modelo_precios.ipynb`, seed fija 42). R² compara cada modelo
con «predecir siempre la media» (M0). **M1 y M2 obtienen R² de test positivo**
(+0.04 y +0.03), algo que no ocurría con la muestra pequeña; **M3 sobreajusta**
(train 0.42 vs. test −0.49) por incorporar demasiadas variables para n=247.


In [2]:
# Tabla 2 · Métricas del split 75/25 (leída del CSV real)
import pandas as pd
from pathlib import Path

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

tabla = pd.read_csv(RAIZ / "reports/tabla_metricas_modelos.csv")
tabla = tabla.rename(columns={"modelo": "Modelo", "n": "n", "R2_train": "R² train",
                              "R2_test": "R² test", "RMSE_train": "RMSE train (USD)",
                              "RMSE_test": "RMSE test (USD)", "MAE_test": "MAE test (USD)"})
tabla.round(2)


,Modelo,n,R² train,R² test,RMSE train (USD),RMSE test (USD),MAE test (USD)
0,M0 baseline,555,0.00,-0.02,20089.28,17598.53,14542.71
1,M1 anio,300,0.14,0.04,14406.63,29636.42,14438.01
2,M2 anio+marca,247,0.35,0.02,18686.88,12635.48,8587.09
3,M3 completo,247,0.42,-0.49,17597.32,15628.11,11510.00


### 4.2 Robustez: validación cruzada 5-fold

Resultados de la sección 5 del notebook `02_modelo_precios.ipynb` (CV 5-fold,
más estable que un único split):

| Modelo | n | R² (media ± desv.) | RMSE (USD) |
|--------|---|--------------------|------------|
| M1 `precio ~ año` | 300 | **−0.085 ± 0.360** | 19 184 |
| M2 `+ marca` | 247 | **−0.378 ± 0.596** | 21 188 |
| M3 `+ municipio + vistas + imágenes` | 247 | **−0.573 ± 0.621** | 22 339 |

La variante logaritmo tampoco generaliza (R² de test ≈ −0.08 a −1.40 en el split;
−0.08 a −0.47 en CV). En conjunto, **la regresión lineal no supera de forma
consistente a predecir la media**, aunque con la muestra ampliada las métricas
mejoraron drásticamente frente a la versión de 150 anuncios (sección 4.4).


### 4.3 Gráficos del modelo

La predicción vs. el precio real muestra que el modelo captura la tendencia del
año pero con errores grandes (RMSE de test ≈ desviación típica de los precios).
Los coeficientes de M2 (base `audi`) ordenan las primas de marca, y el mapa de
métricas resume el sobreajuste de M3.

![Predicción vs. real](../reports/fig_08_prediccion_vs_real.png)

![Coeficientes del modelo 2](../reports/fig_09_coeficientes.png)

![Métricas de los modelos](../reports/fig_10_metricas_modelos.png)

### 4.4 Comparación: muestra de 150 vs. muestra ampliada (Tabla 3)

La ampliación del scraping (150 → 600 anuncios) re-entrenó las Fases 4 y 5. Los
valores de la columna «Antes» provienen de la ejecución previa de
`02_modelo_precios.ipynb` (muestra de 150; respaldo en el historial del proyecto);
«Después» son las salidas de la ejecución actual:

| Métrica (split 75/25, seed 42) | Antes (n=54–68) | Después (n=247–300) |
|--------------------------------|-----------------|---------------------|
| R² test M1 | −0.94 | **+0.04** |
| R² test M2 | −0.88 | **+0.03** |
| R² test M3 | −1.01 | −0.49 |
| CV 5-fold M1 | ≈ −0.14 | −0.09 ± 0.36 |
| CV 5-fold M2 | ≈ −4.4 (sobreajuste) | −0.38 ± 0.60 |
| Efecto del año (M1 completo) | +796 USD/año (r ≈ 0.46) | +716 USD/año (r = 0.29) |

**Lectura:** 4–5× más datos **estabilizaron las métricas** (M1 y M2 pasan a R²
de test positivo; M2 deja de sobreajustar: CV de ≈ −4.4 a −0.38), pero el modelo
**sigue sin generalizar** porque la limitación ya no es el tamaño de muestra sino
la falta de variables fuertes (km, estado, combustible, transmisión están vacías
en Revolico).


### 4.5 Coeficientes de marca (M2 completo, Tabla 4)

Se interpretan del modelo ajustado con toda la muestra de M2 (n=247, R² = 0.312):
la base es `audi` (primera marca alfabética entre las retenidas, `drop="first"`).
Las primas más grandes son **bmw (+51 878 USD)** y **ford (+30 467 USD)** — BMW
está influida por los dos X6 de 120 000 y 250 000 USD, y ford tiene solo 3
anuncios, así que son poco generalizables. Land rover (+18 550) y fiat
(+10 166) también aportan; **porsche sale negativa (−19 633) con solo 4 anuncios**
(coeficiente poco fiable). El R² de la muestra pasa de 0.092 (solo año) a 0.312
(año + marca): **la marca añade señal, pero no suficiente para predecir.**


In [3]:
# Tabla 4 · Coeficientes de M2 completo (leídos del CSV real, base = audi)
import pandas as pd
from pathlib import Path

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

coefs = pd.read_csv(RAIZ / "reports/coeficientes_m2.csv")
coefs["coef"] = coefs["coef"].map("{:+,.0f} USD".format)
coefs.sort_values("coef", ascending=False).reset_index(drop=True)


,feature,coef
0,marca_g_chery,"-7,970 USD"
1,marca_g_kia,"-7,755 USD"
2,marca_g_hyundai,"-5,608 USD"
3,marca_g_peugeot,"-4,833 USD"
4,marca_g_mitsubishi,"-3,071 USD"
5,marca_g_porsche,"-19,633 USD"
6,marca_g_geely,"-12,463 USD"
7,marca_g_toyota,"-1,945 USD"
8,marca_g_nissan,"-1,503 USD"
9,marca_g_foton,"+8,545 USD"


## 5. Discusión y limitaciones

1. **El modelo aún no predice precios de forma útil.** La CV 5-fold es negativa
   en los tres modelos (M1 −0.09, M2 −0.38, M3 −0.57) y el RMSE (~19–22k USD) es
   comparable a la desviación típica de los precios (19.5k USD): ninguna
   especificación supera consistentemente a «predecir la media».
2. **La causa principal es la falta de variables fuertes, no la muestra.**
   Revolico no reporta km, estado, combustible ni transmisión (columnas vacías en
   el 100% de los anuncios) y los precios son extremadamente heterogéneos (1 a
   250 000 USD). Sin esas variables, el techo predictivo es bajo.
3. **M3 sobreajusta** (train 0.42 vs. test −0.49): con n=247, agregar municipio,
   vistas e imágenes añade ruido, no señal.
4. **Sesgos y cautelas:** muestra no aleatoria (solo Revolico La Habana, orden de
   publicación, ≈17% del total), influencia de outliers (BMW X6 a 250 000 USD),
   multicolinealidad marca–año y **correlación ≠ causalidad** (una marca «cara»
   puede reflejar que casi todos sus anuncios son modernos).
5. **Hallazgo didáctico:** ampliar 4–5× la muestra estabilizó y mejoró las
   métricas (evidencia de que el pipeline es correcto), pero demostró que **más
   datos no sustituyen mejores variables** — una conclusión válida y honesta.


## 6. Conclusiones y trabajo futuro

**Conclusiones (con evidencia de las Fases 4 y 5):**

1. **El mercado habanero de anuncios es mayoritariamente moderno** (94% de los
   anuncios con año son 2010+) y está concentrado en pocas marcas (Hyundai 23%).
2. **El año es el predictor más fuerte disponible** (+716 USD/año; r = 0,30),
   consistente entre la muestra de 150 y la de 600.
3. **Existen segmentos de precio por marca:** medio-alto (Toyota, Nissan, Hyundai,
   Kia) frente a económico/de trabajo (Lada, Daewoo); la zona no discrimina.
4. **El modelo de regresión lineal no está listo para predecir** precios nuevos,
   aunque con la muestra ampliada deja de sobreajustar y sus métricas de test se
   vuelven positivas en M1/M2.
5. **Q1 se responde de forma cualitativa:** año y marca son los únicos factores
   con señal detectable en los datos de anuncios.

**Trabajo futuro:**

- **Fase 3 (campo):** recolectar km, estado, combustible y transmisión reales
  (instrumentos ya listos en `docs/instrumentos/` y `data/raw/`) — son las
  variables que certeramente mejorarían el modelo.
- **Scraping enriquecido:** capturar más campos de Revolico u otras fuentes
  (Porlalivre, grupos de Telegram) para llenar el vacío de características.
- **Modelos no lineales** (árboles/random forest) cuando existan más variables.


## 7. Anexos

### A.1 Método de scraping (resumen técnico)

- API descubierta por ingeniería inversa de los *bundles* JS de Revolico
  (`/next/static/chunks/*.js`); introspección GraphQL deshabilitada (ver
  `src/exploracion/`).
- Query `AdsSearch` con `subcategorySlug: "vehiculos-carros"` y
  `provinceSlug: "la-habana"`; paginación `page` + `pageLength=50`.
- Lección clave: GraphQL solo aplica las variables que la query *declara y
  referencia*; si falta `$subcategorySlug`, la query ignora el filtro (bug que
  costó dos ejecuciones en la Fase 2).
- Ampliación: `MIN_ANUNCIOS = 600`, `MAX_PAGES = 16` → 12 páginas efectivas.
- Ética: pausa de 4 s entre peticiones, uso académico, **sin teléfonos ni datos
  de contacto**. Respaldo JSON por página.

### A.2 Glosario

- **R²:** proporción de la varianza del precio explicada por el modelo (1 =
  perfecto, 0 = igual que la media, negativo = peor que la media).
- **RMSE / MAE (USD):** error de predicción medio en dólares.
- **Split 75/25 + seed 42:** 75% entrena, 25% evalúa; semilla fija = reproducible.
- **Validación cruzada 5-fold:** el split se reparte 5 veces y se promedian las
  métricas → estimación más estable.
- **One-Hot (OHE):** codifica la marca en columnas 0/1; ajustado solo en train
  para evitar *data leakage*.
- **log1p(precio):** transformación logarítmica robusta frente a valores
  extremos (precios simbólicos y de lujo).

### A.3 Productos del proyecto

| Producto | Ruta |
|----------|------|
| EDA (Fase 4) | `notebooks/01_eda.ipynb` |
| Modelo de precios (Fase 5) | `notebooks/02_modelo_precios.ipynb` |
| Scraper | `src/scraper_revolico.py` |
| Dataset limpio | `data/processed/mercado_havana.csv` (600 filas) |
| Dataset de regresión | `data/processed/regresion_dataset.csv` (247 filas) |
| Figuras | `reports/fig_01_precios.png` … `fig_10_metricas_modelos.png` |
| Tablas | `reports/tabla_metricas_modelos.csv`, `reports/coeficientes_m2.csv` |
| Documento de estado | `docs/ESTADO_PROYECTO.md` (secciones 6bis y 6ter verificadas) |


---

**Nota metodológica final.** Este informe cita exclusivamente salidas reales de
las ejecuciones registradas en `data/raw/`, `data/processed/`, `reports/` y los
notebooks de las Fases 4 y 5. La muestra original de 150 anuncios (historial de la
revisión) está respaldada en `data/raw/respaldo_150_2026-09-24/` y se usó para la
comparación antes/después de la sección 4.4.

*Proyecto de Ciencia de Datos · 1er año · Universidad de La Habana ·
24 de septiembre de 2026.*
